# Entrega E2 · Tema 4. Random forest y XGBoost para clasificación

**Técnicas Avanzadas de Predicción en Negocios · 05-11-2026**

Datos: Encuesta Cuatrienal de Estructura Salarial 2022 (INE), microdatos. Cada alumno trabaja con una muestra propia de 10 000 trabajadores.

**Cómo está organizada la entrega**

- **Parte A. Código mostrado.** Ejecuta y lee cada celda. Es el patrón que usarás en la parte B.
- **Parte B. Actividad.** Tres bloques con tareas cortas de código que escribes tú, de principio a fin, y una interpretación al final de cada bloque.
- **Cierre.** Una celda guarda tus respuestas en un fichero `.json`. Subes el cuaderno y ese fichero.

Tienes **120 minutos**, incluidos la revisión y la subida. Orientación: 10 min para la preparación, 20 para la parte A, 20 para el bloque 1, 15 para el bloque 2, 20 para el bloque 3 y 15 para el cierre. Quedan unos 20 minutos de margen.

**Coste de cálculo**

- Una búsqueda de hiperparámetros ajusta el modelo con cada combinación de valores en cada una de las cinco particiones de la validación cruzada. Con modelos de cientos de árboles, eso supone muchos ajustes.
- Para que el cuaderno se ejecute en pocos minutos en cualquier ordenador, las búsquedas de esta entrega prueban pocos valores de cada hiperparámetro.
- En un análisis real probaríamos más valores, a costa de más tiempo de cálculo.
- La celda más lenta es la búsqueda del bloque 1.

**Solo si trabajas en Google Colab.** La celda siguiente instala la versión de scikit-learn del entorno `tap` del curso. En tu ordenador, con el entorno `tap` activado, no hace nada. Si Colab pide reiniciar la sesión después de instalar, acepta y vuelve a ejecutar el cuaderno desde el principio.

In [ ]:
# Instala las versiones del curso (solo en Google Colab)
import sys
if "google.colab" in sys.modules:
    %pip install -q -r https://raw.githubusercontent.com/juanDmontoro/tap/main/requirements.txt

## 0. Tu ID

> **Escribe solo el ID de seis cifras que te ha dado el profesor.**
> **No escribas tu nombre, usuario, DNI, NIU ni correo.** El cuaderno no debe contener ningún dato personal.

Tu ID es también tu **semilla**. La semilla fija tu muestra de datos y tus particiones, así que cada alumno obtiene resultados distintos.

In [ ]:
import re

# Escribe entre las comillas el ID de seis cifras que te ha dado el profesor.
ID = "escribe_aqui_tu_id"

# El ID es también tu semilla: un número entero.
ID = ID.strip()
assert re.fullmatch(r"\d{6}", ID), "Escribe tu ID de seis cifras entre las comillas."
SEMILLA = int(ID)
print("ID y semilla:", SEMILLA)

### Preparación

**Ejecuta la celda siguiente.** Carga las bibliotecas y los datos y se queda con los trabajadores que tienen un salario positivo y no superior al percentil 99. Después elige 10 000 trabajadores con tu semilla. No hace falta leerla. Descarga los datos (41 MB), así que tarda entre unos segundos y un par de minutos, según la conexión.

In [ ]:
# Bibliotecas para tratar datos y guardar las respuestas.
import os
import json
import numpy as np
import pandas as pd
import sklearn
import xgboost

# Preparación de los predictores y modelos.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# División, validación cruzada y búsqueda de hiperparámetros.
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV,
)

# Datos de la EES 2022 en la web del curso (separador: tabulación).
URL_DATOS = "https://juandmontoro.github.io/tap/data/EES_2022.tab"
df = pd.read_csv(URL_DATOS, sep="\t", low_memory=False)
df.columns = df.columns.str.replace('"', "").str.strip()

# Trabajadores con salario positivo y no superior al percentil 99.
tope = df["RETRINOIN"].quantile(0.99)
salario_admitido = (df["RETRINOIN"] > 0) & (df["RETRINOIN"] <= tope)
limpio = df[salario_admitido].copy()

# Tu muestra: 10 000 trabajadores elegidos con tu semilla.
datos = limpio.sample(n=10000, random_state=SEMILLA).reset_index(drop=True)
print("Observaciones:", len(datos))

# Núcleos para repartir el trabajo: todos los del ordenador menos uno (al menos uno).
N_NUCLEOS = max(1, os.cpu_count() - 1)
print("Núcleos para los cálculos en paralelo:", N_NUCLEOS)
print("Versiones: scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__)

### Respuesta, predictores y división entre entrenamiento y test

- La respuesta es binaria. Vale 1 (un **positivo**) si el salario bruto anual del trabajador (`RETRINOIN`) supera el **percentil 75 del entrenamiento**, y 0 en otro caso. Así, un 25 % de los trabajadores tienen salario alto.
- El salario no es predictor, porque con él se define la respuesta, y el percentil 75 se calcula solo con el entrenamiento.
- Usamos 16 predictores: 14 categóricos y 2 numéricos, estos últimos como decimales (`float`).
- Reservamos el 25 % de las observaciones para el test. Esta entrega no lo usa, porque todas las elecciones se hacen con validación cruzada en el entrenamiento.

In [ ]:
PRED_CATEGORICOS = [
    "SEXO", "ESTU", "ANOS2", "TIPOCON", "TIPOJOR", "CNO1", "CNACE",
    "CONTROL", "MERCADO", "REGULACION", "RESPONSA", "NUTS1", "TIPOPAIS", "ESTRATO2",
]
PRED_NUMERICOS = ["ANOANTI", "JAP"]

# X contiene los predictores: categóricos como categorías y numéricos como decimales (float).
X = datos[PRED_CATEGORICOS + PRED_NUMERICOS].copy()
X[PRED_CATEGORICOS] = X[PRED_CATEGORICOS].astype("category")
X[PRED_NUMERICOS] = X[PRED_NUMERICOS].astype(float)

# y es el salario, que solo sirve para definir la respuesta.
y = datos["RETRINOIN"]

# Separamos el test antes de ajustar ningún modelo.
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.25,
                                                    random_state=SEMILLA)
print("Entrenamiento:", len(X_train), "| test:", len(X_test))

# Umbral del salario alto: percentil 75 del entrenamiento.
p75 = y_train.quantile(0.75)

# Respuesta binaria: 1 = salario alto, 0 = no.
alto_train = (y_train > p75).astype(int)
print("P75:", round(p75), "euros | proporción de positivos:", round(alto_train.mean(), 3))

### Qué buscamos y cómo lo medimos

Queremos estimar, para cada trabajador, la **probabilidad de salario alto**. Por eso comparamos los modelos con la **log-loss**, que mide la calidad de las probabilidades que predice un modelo.

- Si $p$ es la probabilidad de salario alto que el modelo asigna a un trabajador, la log-loss de ese trabajador vale $-\log(p)$ si tiene salario alto y $-\log(1-p)$ si no lo tiene.
- La log-loss del modelo es la media de esos valores. Cuanto menor, mejor.
- Una probabilidad extrema y equivocada pesa mucho. Si el modelo asigna $p = 0{,}001$ a un trabajador que sí tiene salario alto, ese trabajador suma $-\log(0{,}001) \approx 6{,}9$. Con $p = 0{,}5$ habría sumado $-\log(0{,}5) \approx 0{,}69$.
- La **log-loss de validación cruzada** es la media de las 5 log-loss de las particiones `cv5`, cada una calculada con los trabajadores que esa partición deja fuera del ajuste.
- Su **error típico** es la desviación típica de las 5 log-loss dividida entre √5. Si dos modelos se separan menos que un error típico, la validación cruzada no permite distinguirlos.

Al final de la entrega comparamos los modelos elegidos también con otra métrica, la precisión media, que solo tiene en cuenta el orden en que el modelo coloca a los trabajadores.

---

En la celda siguiente preparamos dos piezas que usan todos los modelos.

- El **preprocesador** crea una columna por categoría de cada predictor categórico y, si se le pide, escala los numéricos. El logit escala los numéricos (`escalar=True`, el valor por defecto). El random forest y XGBoost no necesitan escalar porque un corte de un árbol solo depende del orden de los valores y el escalado no cambia ese orden. Usamos `escalar=False` (si bien podríamos usar el mismo preprocesamiento que en el logit).
- Las particiones `cv5` son las mismas en toda la entrega. `StratifiedKFold` mantiene en cada partición la proporción de salarios altos del entrenamiento.

In [ ]:
# Devuelve un preprocesador nuevo, sin ajustar, en cada llamada.
def crear_preprocesador(escalar=True):
    # Numéricas: escaladas, o sin cambios con "passthrough".
    if escalar:
        numericas = StandardScaler()
    else:
        numericas = "passthrough"
    # Categóricas: una columna por categoría (las de menos del 1 % de los trabajadores se agrupan en una).
    return ColumnTransformer([
        ("numericas", numericas, PRED_NUMERICOS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01,
                                      sparse_output=False), PRED_CATEGORICOS),
    ])


# Cinco particiones del entrenamiento, barajadas con tu semilla, con la misma proporción de salarios altos.
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

# Parte A. Código mostrado

Ejecuta y lee cada celda. Esta parte no tiene preguntas. Es el patrón que adaptarás en la parte B.

## A1. Clasificador de referencia, logit y random forest

Empezamos con dos modelos de referencia, ajustados con tu muestra.

- El **clasificador de referencia** no usa los predictores. Asigna a todos los trabajadores la misma probabilidad de salario alto, la proporción de positivos de su entrenamiento.
- El **modelo base** es el logit, que estima la probabilidad con un modelo lineal en la escala del logaritmo de las odds, $\log\big(p/(1-p)\big)$. Es el modelo sencillo que un modelo más flexible tiene que batir.

> **Python que usamos aquí**
> - `cross_val_score(modelo, X, respuesta, scoring="neg_log_loss", cv=particiones)` devuelve las 5 log-loss de validación, con el signo cambiado.
> - `DummyClassifier(strategy="prior")` crea el clasificador de referencia.
> - `LogisticRegression(max_iter=2000)` crea el logit.

In [ ]:
# Clasificador de referencia: asigna a todos la proporción de positivos de su entrenamiento.
referencia = DummyClassifier(strategy="prior")

puntuaciones_referencia = cross_val_score(referencia,
                                          X_train,
                                          alto_train,
                                          scoring="neg_log_loss",
                                          cv=cv5)

# Modelo base: preprocesador con escalado y logit.
logit = Pipeline([
    ("preprocesado", crear_preprocesador()),
    ("modelo", LogisticRegression(max_iter=2000)),
])

puntuaciones_logit = cross_val_score(logit,
                                     X_train,
                                     alto_train,
                                     scoring="neg_log_loss",
                                     cv=cv5)

# scikit-learn devuelve la log-loss con el signo cambiado: la pasamos a positivo.
print("Clasificador de referencia | log-loss:", round(-puntuaciones_referencia.mean(), 3))
print("Logit | log-loss:", round(-puntuaciones_logit.mean(), 3),
      "| error típico:", round(puntuaciones_logit.std() / np.sqrt(5), 3))

El **random forest** promedia muchos árboles de clasificación.

- Cada árbol se ajusta con un remuestreo bootstrap del entrenamiento, una muestra del mismo tamaño extraída con reemplazamiento, y crece sin límite de profundidad.
- En cada corte, el random forest aleatoriza los predictores. Elige al azar una fracción de las columnas (`max_features`), y el corte solo puede usar esas columnas. La fracción se aplica a las columnas que salen del preprocesador, una por categoría. Así, los árboles se parecen menos entre sí.
- Cada árbol predice como probabilidad de salario alto la proporción de salarios altos de la hoja a la que llega el trabajador. El random forest promedia las probabilidades de sus árboles.
- Con `n_jobs=N_NUCLEOS`, reparte sus árboles entre los núcleos de tu ordenador, con el mismo resultado que con uno solo.

Ajustamos un random forest de 100 árboles que elige entre el 33 % de las columnas en cada corte.

> **Python que usamos aquí**
> - `RandomForestClassifier(n_estimators=B, max_features=fracción, n_jobs=núcleos, random_state=semilla)` crea un random forest de B árboles.

In [ ]:
# Random forest: 100 árboles, y cada corte elige entre el 33 % de las columnas.
bosque = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", RandomForestClassifier(n_estimators=100,
                                      max_features=0.33,
                                      n_jobs=N_NUCLEOS,
                                      random_state=SEMILLA)),
])

puntuaciones_bosque = cross_val_score(bosque,
                                      X_train,
                                      alto_train,
                                      scoring="neg_log_loss",
                                      cv=cv5)

print("Random forest 0,33 | log-loss:", round(-puntuaciones_bosque.mean(), 3),
      "| error típico:", round(puntuaciones_bosque.std() / np.sqrt(5), 3))

Cada árbol crece hasta que sus hojas tienen trabajadores de una sola clase, así que casi siempre predice una probabilidad de 0 o de 1. El random forest promedia esas probabilidades: el promedio es exactamente 0 solo si sus 100 árboles dan 0, y exactamente 1 solo si los 100 dan 1. Contamos cuántas veces ocurre con las predicciones de validación cruzada.

- `cross_val_predict` devuelve la predicción de validación cruzada de cada trabajador, la que le asigna el modelo ajustado sin ese trabajador.

> **Python que usamos aquí**
> - `cross_val_predict(modelo, X, respuesta, cv=particiones, method="predict_proba")` devuelve una fila por trabajador y una columna por clase. La primera columna es la probabilidad de la clase 0 y la segunda, la de la clase 1, el salario alto.

In [ ]:
# Probabilidades de validación cruzada del random forest: una columna por clase.
probabilidades_bosque = cross_val_predict(bosque,
                                          X_train,
                                          alto_train,
                                          cv=cv5,
                                          method="predict_proba")

# [:, 1]: todas las filas de la columna 1, la probabilidad de salario alto.
prob_alto_bosque = probabilidades_bosque[:, 1]

# True si la probabilidad es exactamente 0 o exactamente 1 (el símbolo | significa «o»).
extrema = (prob_alto_bosque == 0) | (prob_alto_bosque == 1)
print("Proporción de trabajadores con probabilidad exactamente 0 o 1:",
      round(extrema.mean(), 3))

Para comparar los tres modelos, reunimos sus log-loss de validación cruzada en una tabla. Todas se calculan con `cross_val_score` y con las mismas particiones `cv5`.

In [ ]:
# Resumen de A1: log-loss de validación cruzada y error típico de las cinco log-loss de cada modelo.
resumen = pd.DataFrame({
    "modelo": ["clasificador de referencia", "logit", "random forest 0,33"],
    "log-loss": [-puntuaciones_referencia.mean(),
                 -puntuaciones_logit.mean(),
                 -puntuaciones_bosque.mean()],
    "error típico": [puntuaciones_referencia.std() / np.sqrt(5),
                     puntuaciones_logit.std() / np.sqrt(5),
                     puntuaciones_bosque.std() / np.sqrt(5)],
})

resumen.round(3)

Para saber qué modelo de la tabla tiene la menor log-loss, buscamos su fila y leemos el nombre de la columna `"modelo"` en esa fila.

> **Python que usamos aquí**
> - `tabla["columna"].idxmin()` devuelve la etiqueta de la fila con el menor valor de esa columna.
> - `tabla["columna"].idxmax()` devuelve la etiqueta de la fila con el mayor valor.

In [ ]:
# Fila de resumen con la menor log-loss de validación cruzada.
fila_menor = resumen["log-loss"].idxmin()

# .loc[fila, "modelo"]: el valor de la columna "modelo" en esa fila.
modelo_menor = resumen.loc[fila_menor, "modelo"]
print("Menor log-loss de validación cruzada:", modelo_menor)

**Lectura de los resultados.**

- El logit y el random forest tienen una log-loss de validación cruzada mucho menor que la del clasificador de referencia, así que sus probabilidades aprovechan la información de los predictores.
- Sin un mínimo de trabajadores por hoja, el random forest suele tener una log-loss de validación cruzada mayor que la del logit, o muy parecida.
- El problema es que una parte de los trabajadores recibe del random forest una probabilidad de exactamente 0 o 1, porque sus 100 árboles le dan esa misma probabilidad. La celda de `cross_val_predict` muestra qué proporción.
- Si alguno de esos trabajadores tiene la clase contraria, la log-loss lo castiga mucho. Con $p = 0$, $-\log(p)$ sería infinito, y scikit-learn le pone un tope muy alto.
- En la parte B construirás árboles cuyas hojas contengan más trabajadores, que dan probabilidades menos extremas, y con otra fracción de columnas por corte.

---

## A2. XGBoost: tasa y profundidad

XGBoost ajusta los árboles uno detrás de otro.

- Cada nuevo árbol intenta corregir los errores de las predicciones acumuladas hasta ese momento.
- Su aportación se multiplica por una **tasa de aprendizaje** ν (`learning_rate`) antes de sumarse a la predicción acumulada. Esta tasa controla cuánto influye cada nuevo árbol.
- En clasificación, las aportaciones de los árboles se acumulan en un índice, como el índice lineal del logit. XGBoost transforma ese índice en una probabilidad entre 0 y 1 con la misma función que el logit.

Vamos a implementar una búsqueda para dos hiperparámetros: la tasa de aprendizaje y la profundidad de cada árbol (`max_depth`). Buscamos la tasa entre 0,03, 0,1 y 0,3 y la profundidad entre 2, 4 y 6, con 300 árboles. Son nueve combinaciones y, como cada una se ajusta en las cinco particiones, 45 ajustes.

> **Python que usamos aquí**
> - `XGBClassifier(n_estimators=B, learning_rate=tasa, max_depth=profundidad, random_state=semilla)` crea un XGBoost de clasificación con la interfaz de scikit-learn ([documentación](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier)).
> - `GridSearchCV(estimator=modelo, param_grid=malla, scoring=métrica, cv=particiones, return_train_score=True, n_jobs=núcleos)` ajusta el modelo con cada combinación de la malla en cada partición. En la malla, el nombre del paso del `Pipeline` y el del hiperparámetro van unidos por dos guiones bajos (`modelo__learning_rate`).

Dos argumentos de `GridSearchCV` añaden información o velocidad.

- Con `return_train_score=True`, `cv_results_` guarda también la log-loss media de entrenamiento (`mean_train_score`). Es la log-loss que el modelo obtiene, en cada partición, con los mismos trabajadores con los que se ha ajustado.
- Con `n_jobs=N_NUCLEOS`, la búsqueda reparte sus ajustes entre los núcleos de tu ordenador.

Definimos el `Pipeline` de XGBoost con 300 árboles, la malla y la búsqueda, y ajustamos la búsqueda con el entrenamiento.

In [ ]:
# XGBoost de clasificación con 300 árboles, sin fijar la tasa ni la profundidad (las prueba la búsqueda).
xgb = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", XGBClassifier(n_estimators=300, random_state=SEMILLA)),
])

# Malla: tres tasas de aprendizaje y tres profundidades.
malla_xgb = {"modelo__learning_rate": [0.03, 0.1, 0.3], "modelo__max_depth": [2, 4, 6]}

# Búsqueda con las particiones cv5. Sus 45 ajustes se reparten entre los núcleos.
busqueda_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=malla_xgb,
    scoring="neg_log_loss",
    cv=cv5,
    return_train_score=True,
    n_jobs=N_NUCLEOS,
)
busqueda_xgb.fit(X_train, alto_train)

Recuperamos la tasa y la profundidad de menor log-loss de validación cruzada.

- `best_params_` guarda esos valores.
- La búsqueda prueba cada combinación en copias de `xgb`, así que `xgb` conserva los valores por defecto de XGBoost: una tasa de 0,3 y una profundidad de 6.
- `best_estimator_` es el `Pipeline` con los valores elegidos y 300 árboles. Lo guardamos en `xgb`, que sustituye al de los valores por defecto. Desde aquí, `xgb` es el XGBoost elegido, el que usarás en la parte B.

In [ ]:
# Tasa y profundidad con menor log-loss de validación cruzada.
tasa = busqueda_xgb.best_params_["modelo__learning_rate"]
profundidad = busqueda_xgb.best_params_["modelo__max_depth"]
print("Tasa:", tasa, "| profundidad:", profundidad)

# Desde aquí, xgb es el Pipeline elegido por búsqueda: tasa, profundidad y 300 árboles.
xgb = busqueda_xgb.best_estimator_

Tabulamos `cv_results_`: las log-loss de validación y de entrenamiento, en positivo, y el error típico.

In [ ]:
# Tabla de cv_results_: log-loss de validación y de entrenamiento en positivo, y error típico.
resultados_xgb = pd.DataFrame(busqueda_xgb.cv_results_)
tabla_xgb = resultados_xgb[["param_modelo__learning_rate",
                            "param_modelo__max_depth",
                            "mean_test_score",
                            "std_test_score",
                            "mean_train_score"]].copy()

tabla_xgb["mean_test_score"] = -tabla_xgb["mean_test_score"]
tabla_xgb["mean_train_score"] = -tabla_xgb["mean_train_score"]
tabla_xgb["error_tipico"] = tabla_xgb["std_test_score"] / np.sqrt(5)

tabla_xgb.round(3)

Leemos en la tabla la fila de la combinación elegida, la que indica `best_index_`, con su log-loss de validación cruzada y su error típico.

In [ ]:
# Fila de la combinación elegida en tabla_xgb, y su log-loss y su error típico.
fila_xgb = busqueda_xgb.best_index_
logloss_xgb = tabla_xgb.loc[fila_xgb, "mean_test_score"]
et_xgb = tabla_xgb.loc[fila_xgb, "error_tipico"]
print("XGBoost elegido | log-loss:", round(logloss_xgb, 3),
      "| error típico:", round(et_xgb, 3))

**Lectura de los resultados.**

- Guardamos los valores de menor log-loss de validación cruzada en `tasa` y `profundidad`, y el `Pipeline` con esos valores en `xgb`. Los usarás en la parte B.
- Al subir la tasa o la profundidad, la log-loss de entrenamiento baja siempre, porque cada árbol corrige una parte mayor de los errores y combina más predictores.
- Sin embargo, la log-loss de validación cruzada baja hasta un mínimo y después vuelve a subir.
- Con ν = 0,3 y profundidad 6, la log-loss de entrenamiento es la menor de la tabla y la de validación cruzada, la mayor, y esa distancia entre las dos es el indicio de sobreajuste.
- Si la segunda mejor combinación queda a menos de un error típico de la mínima, la validación cruzada no permite distinguirlas.
- En la mayoría de las muestras, el XGBoost elegido tiene una log-loss de validación cruzada (`logloss_xgb`) menor que la del logit de `resumen`. Si la diferencia es menor que un error típico, la validación cruzada no permite distinguir los dos modelos.

---

# Parte B. Actividad

Cada bloque plantea un problema dividido en tareas cortas (T). En cada tarea:

- Escribe el código completo en la celda «Escribe aquí tu solución».
- Crea los objetos **con los nombres exactos** que indica el enunciado. La corrección los busca por su nombre.
- Adapta el código de la sección de la parte A que se indica.

Si una tarea no te sale, sigue con la siguiente. Cada tarea se puntúa por separado.

Las interpretaciones se escriben entre las comillas triples `"""` de su celda. Responde a lo que se pregunta, **con tus cifras**.

## Bloque 1. Random forest: tamaño de las hojas y columnas por corte

En A1, el random forest no fijaba un mínimo de trabajadores por hoja (`min_samples_leaf=1`) y elegía entre el 33 % de las columnas en cada corte. Ahora buscamos los dos hiperparámetros a la vez, con una malla de dos dimensiones como la de A2.

- `min_samples_leaf` es el número mínimo de trabajadores de cada hoja. Con hojas de más trabajadores, cada árbol da probabilidades menos extremas.
- Con menos columnas por corte (`max_features` menor), los árboles se parecen menos entre sí.

**T1.1.** Crea `busqueda_b1`, un `GridSearchCV` con el `Pipeline` `bosque` de A1, la malla `{"modelo__min_samples_leaf": [1, 2, 5], "modelo__max_features": [0.1, 0.2, 0.33]}` (sustituye los valores de `bosque`), la métrica `"neg_log_loss"`, las particiones `cv5` y `return_train_score=True`, y ajústalo con el entrenamiento. Es como `busqueda_xgb` en A2, pero sin `n_jobs`, porque el random forest ya reparte sus árboles entre los núcleos. Son 9 combinaciones y 45 ajustes, así que esta celda es la más lenta de la entrega.

In [ ]:
# T1.1 · busqueda_b1
# Escribe aquí tu solución


**T1.2.** Construye la tabla `tabla_b1` con las columnas `param_modelo__min_samples_leaf`, `param_modelo__max_features`, `mean_test_score` (la log-loss de validación cruzada, en positivo), `std_test_score`, `mean_train_score` (la log-loss de entrenamiento, en positivo) y `error_tipico` (como `tabla_xgb` en A2).

In [ ]:
# T1.2 · tabla_b1
# Escribe aquí tu solución


**T1.3.** Guarda en `msl_elegido_b1` y `mf_elegido_b1` el `min_samples_leaf` y el `max_features` de menor log-loss de validación cruzada (los números que guarda `best_params_`), en `logloss_rf_b1` esa log-loss, que debe ser **positiva**, y en `et_rf_b1` su error típico (la fila `best_index_` de `tabla_b1`, como `fila_xgb` en A2).

In [ ]:
# T1.3 · msl_elegido_b1, mf_elegido_b1, logloss_rf_b1 y et_rf_b1
# Escribe aquí tu solución


**Interpretación 1.** Usa tu tabla `tabla_b1`.

1. ¿Qué combinación tiene la menor log-loss de validación cruzada? ¿Mejora al random forest de A1 (`min_samples_leaf=1` y `max_features=0.33`) por más de un error típico? Da las cifras.
2. En la fila del random forest de A1, ¿cuánto valen la log-loss de entrenamiento y la de validación cruzada, y qué indica la diferencia? ¿Cómo cambia la log-loss de entrenamiento al subir `min_samples_leaf`?

In [ ]:
interp_b1 = """
Escribe aquí tu respuesta.
"""

## Bloque 2. XGBoost: ¿cuántos árboles?

En A2, los modelos tenían 300 árboles. Ahora buscamos el número de árboles con validación cruzada, con la `tasa` y `profundidad` elegidas en A2.

**T2.1.** Crea `busqueda_b2`, un `GridSearchCV` con el `Pipeline` `xgb` de A2, que ya tiene la tasa y la profundidad elegidas, la malla `{"modelo__n_estimators": [50, 100, 200, 400]}` (sustituye sus 300 árboles), la métrica `"neg_log_loss"`, las particiones `cv5`, `return_train_score=True` y `n_jobs=N_NUCLEOS`, y ajústalo con el entrenamiento (como `busqueda_xgb` en A2).

In [ ]:
# T2.1 · busqueda_b2
# Escribe aquí tu solución


**T2.2.** Construye la tabla `tabla_b2` con las columnas `param_modelo__n_estimators`, `mean_test_score` y `mean_train_score` (en positivo), `std_test_score` y `error_tipico` (como `tabla_xgb` en A2).

In [ ]:
# T2.2 · tabla_b2
# Escribe aquí tu solución


**T2.3.** Guarda en `arboles_min_b2` el número de árboles de menor log-loss de validación cruzada (el que guarda `best_params_`), en `logloss_xgb_b2` esa log-loss, **positiva**, y en `et_xgb_b2` su error típico (como en T1.3).

In [ ]:
# T2.3 · arboles_min_b2, logloss_xgb_b2 y et_xgb_b2
# Escribe aquí tu solución


**Interpretación 2.** Usa tu tabla `tabla_b2`.

1. ¿Cómo cambian la log-loss de entrenamiento y la de validación cruzada al pasar de 50 a 400 árboles? ¿Hay indicio de sobreajuste? Da las cifras.
2. Compara la log-loss de validación cruzada del número de árboles elegido (`logloss_xgb_b2`) con la del XGBoost de 300 árboles de A2 (`logloss_xgb`). ¿Mejora elegir el número de árboles por más de un error típico?

In [ ]:
interp_b2 = """
Escribe aquí tu respuesta.
"""

## Bloque 3. Dos métricas: log-loss y precisión media

Hemos elegido los modelos con la log-loss, porque queremos estimar la probabilidad de salario alto de cada trabajador. Ahora comparamos los modelos elegidos también con la **precisión media** (AP, *average precision*).

- Ordenamos a los trabajadores de mayor a menor probabilidad de salario alto y los seleccionamos en ese orden. La **precisión** es la proporción de salarios altos entre los seleccionados, y la **sensibilidad**, la proporción de todos los salarios altos que ya hemos seleccionado.
- La AP resume la curva de precisión frente a sensibilidad. Es la media de la precisión en cada momento en que la selección incluye a un trabajador con salario alto. Va de 0 a 1, y cuanto mayor, mejor.
- La AP solo depende del orden de las probabilidades. Si un modelo divide todas sus probabilidades entre dos, el orden de los trabajadores no cambia y su AP tampoco. Su log-loss, en cambio, empeora si sus probabilidades eran buenas.
- El clasificador de referencia da a todos la misma probabilidad y no ordena a nadie. Su AP es la proporción de positivos, 0,25.
- En `cross_val_score`, la AP se pide con `scoring="average_precision"`, y scikit-learn la devuelve con su signo, porque una AP mayor es mejor.

**T3.1.** Calcula con `cross_val_score`, `scoring="average_precision"` y las particiones `cv5` las cinco AP de validación de cuatro modelos (como en A1).

- `ap_referencia_b3`, con el clasificador `referencia` de A1.
- `ap_logit_b3`, con el `logit` de A1.
- `ap_rf_b3`, con el random forest elegido en el bloque 1 (`busqueda_b1.best_estimator_`).
- `ap_xgb_b3`, con el XGBoost elegido en el bloque 2 (`busqueda_b2.best_estimator_`).

Si no has completado el bloque 1 o el 2, usa en su lugar `bosque` o `xgb` de la parte A.

In [ ]:
# T3.1 · ap_referencia_b3, ap_logit_b3, ap_rf_b3 y ap_xgb_b3
# Escribe aquí tu solución


**T3.2.** Crea `tabla_b3`, con una fila por modelo, con estos nombres exactos (`"clasificador de referencia"`, `"logit"`, `"random forest elegido en B1"` y `"XGBoost elegido en B2"`), y las columnas `"modelo"`, `"log-loss"`, `"error típico log-loss"`, `"AP"` y `"error típico AP"` (como `resumen` en A1).

- La log-loss y su error típico salen de `puntuaciones_referencia` y `puntuaciones_logit` de A1 y de tus `logloss_rf_b1`, `et_rf_b1`, `logloss_xgb_b2` y `et_xgb_b2`.
- La AP es la media de las cinco AP de T3.1, y su error típico, su desviación típica dividida entre √5.
- Si no has completado el bloque 1 o el 2, usa en esa fila las cifras del random forest 0,33 de A1 (`puntuaciones_bosque`) o las del XGBoost de A2 (`logloss_xgb` y `et_xgb`).

In [ ]:
# T3.2 · tabla_b3
# Escribe aquí tu solución


**T3.3.** Guarda en `modelo_logloss_b3` el nombre (columna `"modelo"`) de la fila de `tabla_b3` con menor log-loss, y en `modelo_ap_b3`, el de la fila con mayor AP (como `modelo_menor` en A1, con `idxmin()` para la log-loss e `idxmax()` para la AP).

In [ ]:
# T3.3 · modelo_logloss_b3 y modelo_ap_b3
# Escribe aquí tu solución


**Interpretación 3.** Usa tu tabla `tabla_b3`.

1. ¿Qué modelo tiene la menor log-loss y cuál la mayor AP? ¿Ordenan igual las dos métricas al logit, al random forest y a XGBoost?
2. Con cada métrica, ¿la diferencia entre el primer modelo y el segundo supera el error típico del primero? Da las cifras.
3. Queremos estimar la probabilidad de salario alto de cada trabajador. ¿Con qué métrica elegirías el modelo, y qué modelo eliges? ¿En qué caso elegirías con la AP?

In [ ]:
interp_b3 = """
Escribe aquí tu respuesta.
"""

# Cierre: guarda tus respuestas

1. Reinicia el kernel y ejecuta todas las celdas: en VS Code, botones **Restart** y **Run All** de la barra superior del cuaderno. En Jupyter, el icono ⏩ de la barra (reiniciar el kernel y ejecutar todo). El cuaderno debe ejecutarse de principio a fin. Tarda unos minutos.
2. Comprueba que la celda siguiente no señala ninguna respuesta pendiente.
3. Sube a Aula Virtual **dos ficheros**: este cuaderno (`.ipynb`) y `entrega_E2_<tu ID>.json`, que se crea en la misma carpeta que el cuaderno.

**No modifiques esta celda.**

In [ ]:
# Objetos que se corrigen, por su nombre.
NOMBRES = [
    "tabla_b1", "msl_elegido_b1", "mf_elegido_b1",
    "logloss_rf_b1", "et_rf_b1", "interp_b1",
    "tabla_b2", "arboles_min_b2", "logloss_xgb_b2", "et_xgb_b2", "interp_b2",
    "tabla_b3", "modelo_logloss_b3", "modelo_ap_b3", "interp_b3",
]

# Resultados de la parte A que acompañan a la corrección.
NOMBRES_A = ["p75", "resumen", "tabla_xgb", "tasa", "profundidad",
             "logloss_xgb", "et_xgb"]

# Columnas que deben tener las tablas de la parte B.
COLUMNAS = {
    "tabla_b1": ["param_modelo__min_samples_leaf", "param_modelo__max_features",
                 "mean_test_score", "std_test_score", "mean_train_score", "error_tipico"],
    "tabla_b2": ["param_modelo__n_estimators", "mean_test_score", "std_test_score",
                 "mean_train_score", "error_tipico"],
    "tabla_b3": ["modelo", "log-loss", "error típico log-loss", "AP", "error típico AP"],
}

# Nombres de los modelos que debe tener la columna "modelo" de tabla_b3.
MODELOS_B3 = ["clasificador de referencia",
              "logit",
              "random forest elegido en B1",
              "XGBoost elegido en B2"]

def a_python(valor):
    # Convierte tablas y objetos de numpy en tipos que admite JSON.
    if isinstance(valor, pd.DataFrame):
        return json.loads(valor.to_json(orient="records"))
    if hasattr(valor, "tolist"):
        return valor.tolist()
    return valor

respuestas = {nombre: a_python(globals().get(nombre)) for nombre in NOMBRES + NOMBRES_A}

# Avisos de columnas que faltan en las tablas de la parte B.
avisos = []
for nombre, columnas in COLUMNAS.items():
    tabla = globals().get(nombre)
    if isinstance(tabla, pd.DataFrame):
        faltan = [columna for columna in columnas if columna not in tabla.columns]
        if faltan:
            avisos.append(f"{nombre}: faltan las columnas {faltan}")

# Aviso si los nombres de la columna "modelo" de tabla_b3 no son los del enunciado.
tabla = globals().get("tabla_b3")
if isinstance(tabla, pd.DataFrame) and "modelo" in tabla.columns:
    if sorted(tabla["modelo"].astype(str)) != sorted(MODELOS_B3):
        avisos.append(f"tabla_b3: la columna modelo debe tener los nombres {MODELOS_B3}")

respuestas.update({
    "semilla": SEMILLA,
    "avisos": avisos,
    "sklearn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "numpy": np.__version__,
})

fichero = f"entrega_E2_{ID}.json"
with open(fichero, "w", encoding="utf-8") as f:
    json.dump(respuestas, f, ensure_ascii=False, indent=2, default=str)

pendientes = [n for n in NOMBRES
              if respuestas[n] is None
              or (n.startswith("interp")
                  and "Escribe aquí tu respuesta" in str(respuestas[n]))]
print("Guardado:", fichero)
print("Respuestas pendientes:", pendientes if pendientes else "ninguna")
print("Avisos:", avisos if avisos else "ninguno")